In [1]:
import mlflow

mlflow.set_tracking_uri("file:./mlruns")

In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

c:\Users\Soham\Documents\youtube comment analyzer\yt_env\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location=('file:c:/Users/Soham/Documents/youtube comment '
 'analyzer/mlruns/901947899065852447'), creation_time=1778952120616, experiment_id='901947899065852447', last_update_time=1778952120616, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna


In [4]:
df = pd.read_csv('reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [ ]:
# Step 1: (Optional) Remapping - skipped since not strictly needed for Multinomial Naive Bayes

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for Multinomial Naive Bayes
def objective_mnb(trial):
    alpha = trial.suggest_float('alpha', 1e-4, 1.0, log=True)  # Tuning the smoothing parameter

    # MultinomialNB model setup
    model = MultinomialNB(alpha=alpha)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for Multinomial Naive Bayes, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_mnb, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = MultinomialNB(alpha=best_params['alpha'])

    # Log the best model with MLflow, passing the algo_name as "MultinomialNB"
    log_mlflow("MultinomialNB", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for Multinomial Naive Bayes
run_optuna_experiment()


[I 2024-10-11 06:08:30,687] A new study created in memory with name: no-name-9da59cb1-1dcb-42a2-baa0-edcf7788ad87
[I 2024-10-11 06:08:30,706] Trial 0 finished with value: 0.6682519551891778 and parameters: {'alpha': 0.026880491877817547}. Best is trial 0 with value: 0.6682519551891778.
[I 2024-10-11 06:08:30,723] Trial 1 finished with value: 0.6650813781441556 and parameters: {'alpha': 0.7921742309070803}. Best is trial 0 with value: 0.6682519551891778.
[I 2024-10-11 06:08:30,740] Trial 2 finished with value: 0.6682519551891778 and parameters: {'alpha': 0.007265133055507888}. Best is trial 0 with value: 0.6682519551891778.
[I 2024-10-11 06:08:30,760] Trial 3 finished with value: 0.6682519551891778 and parameters: {'alpha': 0.05148571972421229}. Best is trial 0 with value: 0.6682519551891778.
[I 2024-10-11 06:08:30,778] Trial 4 finished with value: 0.6682519551891778 and parameters: {'alpha': 0.010280941966723727}. Best is trial 0 with value: 0.6682519551891778.
[I 2024-10-11 06:08:30,7

In [5]:
# =========================================================
# IMPORTS
# =========================================================
import mlflow
import mlflow.sklearn
import optuna
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score
)

from sklearn.naive_bayes import MultinomialNB

from imblearn.over_sampling import SMOTE

# =========================================================
# RANDOM SEED
# =========================================================
np.random.seed(42)

# =========================================================
# REMOVE NaN TARGETS
# =========================================================
df = df.dropna(
    subset=['category']
)

# =========================================================
# TRAIN TEST SPLIT FIRST
# =========================================================
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# =========================================================
# TF-IDF SETTINGS
# =========================================================
ngram_range = (1, 3)

max_features = 1000

# =========================================================
# TF-IDF VECTORIZATION
# =========================================================
vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

# FIT ONLY ON TRAIN
X_train_vec = vectorizer.fit_transform(
    X_train_text
)

# TRANSFORM TEST
X_test_vec = vectorizer.transform(
    X_test_text
)

# =========================================================
# APPLY SMOTE ONLY ON TRAIN DATA
# =========================================================
smote = SMOTE(
    random_state=42
)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_vec,
    y_train
)

# =========================================================
# OPTUNA OBJECTIVE FUNCTION
# =========================================================
def objective_mnb(trial):

    alpha = trial.suggest_float(
        'alpha',
        1e-4,
        10.0,
        log=True
    )

    # -----------------------------------------------------
    # MODEL
    # -----------------------------------------------------
    model = MultinomialNB(
        alpha=alpha
    )

    # -----------------------------------------------------
    # TRAIN
    # -----------------------------------------------------
    model.fit(
        X_train_resampled,
        y_train_resampled
    )

    # -----------------------------------------------------
    # PREDICT
    # -----------------------------------------------------
    y_pred = model.predict(
        X_test_vec
    )

    # -----------------------------------------------------
    # METRIC
    # -----------------------------------------------------
    macro_f1 = f1_score(
        y_test,
        y_pred,
        average='macro'
    )

    return macro_f1


# =========================================================
# RUN OPTUNA
# =========================================================
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective_mnb,
    n_trials=30
)

# =========================================================
# BEST PARAMETERS
# =========================================================
print("=" * 60)

print("BEST PARAMETERS")

print(study.best_params)

print("=" * 60)

# =========================================================
# BEST MODEL
# =========================================================
best_model = MultinomialNB(

    alpha=study.best_params['alpha']
)

# =========================================================
# TRAIN BEST MODEL
# =========================================================
best_model.fit(
    X_train_resampled,
    y_train_resampled
)

# =========================================================
# PREDICTIONS
# =========================================================
y_pred = best_model.predict(
    X_test_vec
)

# =========================================================
# METRICS
# =========================================================
accuracy = accuracy_score(
    y_test,
    y_pred
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average='macro'
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average='weighted'
)

print(f"Accuracy: {accuracy:.4f}")

print(f"Macro F1: {macro_f1:.4f}")

print(f"Weighted F1: {weighted_f1:.4f}")

# =========================================================
# CLASSIFICATION REPORT
# =========================================================
print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred
    )
)

# =========================================================
# MLFLOW LOGGING
# =========================================================
with mlflow.start_run():

    # -----------------------------------------------------
    # TAGS
    # -----------------------------------------------------
    mlflow.set_tag(
        "mlflow.runName",
        "MultinomialNB_SMOTE_TFIDF_Trigram"
    )

    mlflow.set_tag(
        "experiment_type",
        "algorithm_comparison"
    )

    # -----------------------------------------------------
    # PARAMETERS
    # -----------------------------------------------------
    mlflow.log_param(
        "vectorizer",
        "TF-IDF"
    )

    mlflow.log_param(
        "ngram_range",
        ngram_range
    )

    mlflow.log_param(
        "max_features",
        max_features
    )

    mlflow.log_params(
        study.best_params
    )

    # -----------------------------------------------------
    # METRICS
    # -----------------------------------------------------
    mlflow.log_metric(
        "accuracy",
        accuracy
    )

    mlflow.log_metric(
        "macro_f1",
        macro_f1
    )

    mlflow.log_metric(
        "weighted_f1",
        weighted_f1
    )

    # -----------------------------------------------------
    # CLASSIFICATION REPORT
    # -----------------------------------------------------
    report = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, metrics in report.items():

        if isinstance(metrics, dict):

            for metric_name, metric_value in metrics.items():

                mlflow.log_metric(
                    f"{label}_{metric_name}",
                    metric_value
                )

    # -----------------------------------------------------
    # SAVE MODEL
    # -----------------------------------------------------
    mlflow.sklearn.log_model(
        best_model,
        "multinomial_nb_model"
    )

[I 2026-05-17 14:00:05,372] A new study created in memory with name: no-name-7b7f9e07-6e7a-4b3a-b1a7-ab0b4d60c4bb
[I 2026-05-17 14:00:05,440] Trial 0 finished with value: 0.6335005254907907 and parameters: {'alpha': 3.093609850342397}. Best is trial 0 with value: 0.6335005254907907.
[I 2026-05-17 14:00:05,475] Trial 1 finished with value: 0.6351506729621467 and parameters: {'alpha': 2.4924239996392306}. Best is trial 1 with value: 0.6351506729621467.
[I 2026-05-17 14:00:05,511] Trial 2 finished with value: 0.6419174974603151 and parameters: {'alpha': 1.0461414270829266}. Best is trial 2 with value: 0.6419174974603151.
[I 2026-05-17 14:00:05,543] Trial 3 finished with value: 0.6446968787547299 and parameters: {'alpha': 0.00010134202148655462}. Best is trial 3 with value: 0.6446968787547299.
[I 2026-05-17 14:00:05,582] Trial 4 finished with value: 0.6442478229776835 and parameters: {'alpha': 0.008045560137981052}. Best is trial 3 with value: 0.6446968787547299.
[I 2026-05-17 14:00:05,618

BEST PARAMETERS
{'alpha': 0.00010134202148655462}
Accuracy: 0.6604
Macro F1: 0.6447
Weighted F1: 0.6672

Classification Report:

              precision    recall  f1-score   support

          -1       0.46      0.66      0.54      1650
           0       0.71      0.57      0.64      2529
           1       0.79      0.73      0.76      3154

    accuracy                           0.66      7333
   macro avg       0.65      0.65      0.64      7333
weighted avg       0.69      0.66      0.67      7333



2026/05/17 14:00:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 14:00:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
